#  Évaluation Hors Domaine — Domain Shift

**Projet 3 — Détection automatique de Fake News politiques**

Ce notebook évalue la **capacité de généralisation** des modèles sur le BuzzFeed Political Dataset (jamais vu pendant l'entraînement).

### Questions auxquelles on répond :
- Comment les performances évoluent-elles hors du domaine d'entraînement ?
- Quelles sont les causes du domain shift ?
- Quel modèle généralise le mieux ?

## 0. Imports & Configuration

In [ ]:
import os, re, json, joblib
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import nltk
nltk.download('stopwords', quiet=True)
from nltk.corpus import stopwords
from sentence_transformers import SentenceTransformer
from sklearn.metrics import (
    classification_report, f1_score,
    accuracy_score, confusion_matrix
)
import warnings
warnings.filterwarnings('ignore')

sns.set_theme(style='whitegrid')
STOP_WORDS  = set(stopwords.words('english'))
BUZZFEED_URL = (
    'https://raw.githubusercontent.com/BuzzFeedNews/'
    '2016-10-facebook-fact-check/master/data/facebook-fact-check.csv'
)
print(' Imports OK')

## 1. Chargement des modèles entraînés sur LIAR

In [ ]:
tfidf    = joblib.load('../data/modeles/tfidf_vectorizer.joblib')
lr_model = joblib.load('../data/modeles/baseline_logreg.joblib')
sbert_clf= joblib.load('../data/modeles/sbert_best_clf.joblib')
sbert    = SentenceTransformer('all-MiniLM-L6-v2')

# Scores in-domain depuis le JSON
with open('../data/modeles/all_models_metrics.json') as f:
    all_metrics = json.load(f)
metrics_dict = {m['Modèle']: m for m in all_metrics}

lr_f1_in    = metrics_dict.get('Logistic Regression', {}).get('F1 Macro', 0)
sbert_entry = [m for m in all_metrics if 'SBERT' in m['Modèle'] and 'Logistic' in m['Modèle']]
sbert_f1_in = sbert_entry[0]['F1 Macro'] if sbert_entry else 0

print(f' Modèles chargés')
print(f'  LR  in-domain F1 = {lr_f1_in:.3f}')
print(f'  SBERT in-domain F1 = {sbert_f1_in:.3f}')

## 2. Chargement & Préparation du BuzzFeed Dataset

In [ ]:
print(' Chargement du BuzzFeed Political Dataset...')
try:
    buzz_df = pd.read_csv(BUZZFEED_URL)
    print(f' {len(buzz_df)} articles chargés')
    print(f'Colonnes : {buzz_df.columns.tolist()}')
    print(f'\nDistribution des labels :')
    display(buzz_df['Rating'].value_counts())
except Exception as e:
    print(f' {e}')
    print('→ Télécharge depuis : https://github.com/BuzzFeedNews/2016-10-facebook-fact-check')

In [ ]:
# Mapping BuzzFeed → binaire
# Justification : même logique que LIAR — contenu trompeur vs vrai
BUZZ_MAP = {
    'mostly true':                1,
    'mixture of true and false':  1,
    'mostly false':               0,
    'no factual content':         0,
}
buzz_df['binary_label'] = buzz_df['Rating'].str.lower().map(BUZZ_MAP)
buzz_df = buzz_df.dropna(subset=['binary_label','title']).copy()
buzz_df['binary_label']  = buzz_df['binary_label'].astype(int)
buzz_df['enriched_text'] = buzz_df['title'].fillna('')

y_buzz = buzz_df['binary_label'].values

print(f'Après nettoyage : {len(buzz_df)} articles')
print(f'  fake (0) : {(y_buzz==0).sum()}')
print(f'  real (1) : {(y_buzz==1).sum()}')

## 3. Évaluation out-of-domain

In [ ]:
# Logistic Regression
print(' Logistic Regression (TF-IDF) sur BuzzFeed...')
X_buzz_tfidf = tfidf.transform(buzz_df['enriched_text'])
y_buzz_lr    = lr_model.predict(X_buzz_tfidf)
buzz_lr_f1   = f1_score(y_buzz, y_buzz_lr, average='macro')
buzz_lr_acc  = accuracy_score(y_buzz, y_buzz_lr)
print(f'  Accuracy={buzz_lr_acc:.3f} | F1 macro={buzz_lr_f1:.3f}')

# SBERT
print('\n SBERT sur BuzzFeed (~1 min)...')
X_buzz_emb   = sbert.encode(buzz_df['enriched_text'].tolist(),
                              batch_size=64, show_progress_bar=True)
y_buzz_sbert  = sbert_clf.predict(X_buzz_emb)
buzz_sbert_f1 = f1_score(y_buzz, y_buzz_sbert, average='macro')
buzz_sbert_acc= accuracy_score(y_buzz, y_buzz_sbert)
print(f'  Accuracy={buzz_sbert_acc:.3f} | F1 macro={buzz_sbert_f1:.3f}')

In [ ]:
print('='*60)
print('        RÉSULTATS DOMAIN SHIFT')
print('='*60)
ds_data = [
    {'Modèle': 'Logistic Reg. (TF-IDF)', 'In-domain': lr_f1_in,
     'Out-domain': buzz_lr_f1,  'Δ F1': buzz_lr_f1 - lr_f1_in},
    {'Modèle': 'SBERT + LR',             'In-domain': sbert_f1_in,
     'Out-domain': buzz_sbert_f1, 'Δ F1': buzz_sbert_f1 - sbert_f1_in},
]
ds_df = pd.DataFrame(ds_data)
display(ds_df.style.format({'In-domain': '{:.3f}', 'Out-domain': '{:.3f}', 'Δ F1': '{:+.3f}'}))

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(17, 5))

# Grouped bar
models_ds  = ['LR (TF-IDF)', 'SBERT']
in_scores  = [lr_f1_in, sbert_f1_in]
out_scores = [buzz_lr_f1, buzz_sbert_f1]
x = np.arange(len(models_ds))
axes[0].bar(x-0.2, in_scores,  0.35, label='In-domain (LIAR)',        color='#7F77DD')
axes[0].bar(x+0.2, out_scores, 0.35, label='Out-of-domain (BuzzFeed)', color='#EF9F27')
axes[0].set_xticks(x)
axes[0].set_xticklabels(models_ds)
axes[0].set_ylim(0, 1)
axes[0].set_title('F1 macro — Domain Shift', fontweight='bold')
axes[0].legend(fontsize=9)

# Delta
deltas = [o-i for i,o in zip(in_scores, out_scores)]
axes[1].bar(models_ds, deltas,
            color=['#E24B4A' if d<0 else '#1D9E75' for d in deltas],
            edgecolor='white')
axes[1].axhline(0, color='gray', linestyle='--', alpha=0.7)
axes[1].set_title('Δ F1 (out − in)', fontweight='bold')
for i,d in enumerate(deltas):
    axes[1].text(i, d+(0.005 if d>=0 else -0.018), f'{d:+.3f}',
                 ha='center', fontweight='bold')

# Matrices OOD côte à côte
cm1 = confusion_matrix(y_buzz, y_buzz_lr)
cm2 = confusion_matrix(y_buzz, y_buzz_sbert)
cm_combined = np.block([[cm1, np.zeros((2,1), dtype=int)],
                         [np.zeros((1,3), dtype=int)]])
sns.heatmap(cm1, annot=True, fmt='d', cmap='Oranges', ax=axes[2],
            xticklabels=['fake','real'], yticklabels=['fake','real'], cbar=False)
axes[2].set_title(f'LR BuzzFeed\nF1={buzz_lr_f1:.3f}', fontweight='bold')
axes[2].set_xlabel('Prédit')
axes[2].set_ylabel('Vrai')

plt.suptitle('Analyse du Domain Shift — LIAR → BuzzFeed', fontweight='bold')
plt.tight_layout()
plt.savefig('../Doc/OOD_01_domain_shift.png', dpi=150, bbox_inches='tight')
plt.show()

## 4. Analyse des causes du domain shift

In [ ]:
# Comparaison longueur textes LIAR vs BuzzFeed
liar_test_df = pd.read_parquet('../data/traitees/liar_test.parquet')

liar_len = liar_test_df['statement'].fillna('').apply(lambda x: len(str(x).split()))
buzz_len = buzz_df['title'].fillna('').apply(lambda x: len(str(x).split()))

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

axes[0].hist(liar_len, bins=30, alpha=0.7, color='#7F77DD', label=f'LIAR (moy={liar_len.mean():.1f})', edgecolor='white')
axes[0].hist(buzz_len, bins=30, alpha=0.7, color='#EF9F27', label=f'BuzzFeed (moy={buzz_len.mean():.1f})', edgecolor='white')
axes[0].set_title('Longueur des textes : LIAR vs BuzzFeed', fontweight='bold')
axes[0].set_xlabel('Nombre de mots')
axes[0].set_ylabel('Fréquence')
axes[0].legend()

# Distribution des labels
liar_dist = pd.Series(liar_test_df['binary_label'].map({0:'fake',1:'real'}).value_counts())
buzz_dist = pd.Series(pd.Series(y_buzz).map({0:'fake',1:'real'}).value_counts())

x = np.arange(2)
axes[1].bar(x-0.2, [liar_dist.get('fake',0)/len(liar_test_df)*100,
                     liar_dist.get('real',0)/len(liar_test_df)*100],
            0.35, label='LIAR', color='#7F77DD')
axes[1].bar(x+0.2, [buzz_dist.get('fake',0)/len(buzz_df)*100,
                     buzz_dist.get('real',0)/len(buzz_df)*100],
            0.35, label='BuzzFeed', color='#EF9F27')
axes[1].set_xticks(x)
axes[1].set_xticklabels(['fake','real'])
axes[1].set_title('Distribution des labels (%)', fontweight='bold')
axes[1].set_ylabel('%')
axes[1].legend()

plt.tight_layout()
plt.savefig('../Doc/OOD_02_liar_vs_buzzfeed.png', dpi=150, bbox_inches='tight')
plt.show()

print('Causes identifiées du domain shift :')
print(f'  ① Longueur : LIAR moy={liar_len.mean():.1f} mots vs BuzzFeed moy={buzz_len.mean():.1f} mots')
print('  ② Style : déclarations directes vs titres journalistiques')
print('  ③ Vocabulaire : politique US 2015+ vs Facebook 2016')
print('  ④ Format : phrase vs headline condensé')

## 5. Sauvegarde

In [ ]:
buzz_df['pred_lr']    = y_buzz_lr
buzz_df['pred_sbert'] = y_buzz_sbert
buzz_df.to_csv('../data/traitees/buzzfeed_with_preds.csv', index=False)

ds_df.to_json('../data/modeles/domain_shift_results.json', orient='records', indent=2)

print(' data/traitees/buzzfeed_with_preds.csv')
print(' data/modeles/domain_shift_results.json')
print('\n Évaluation hors domaine terminée → lancer Interpretabilite_Biais.ipynb')